# Лабораторна робота: Pandas + API (HTTP-запити та JSON)

Отримання, обробка та збереження даних з REST API за допомогою Pandas.

---
**Рівень:** Junior Data Engineer  
**Що вивчите:** requests, JSON parsing, pd.json_normalize, API pagination, робота з реальними API

In [ ]:
import pandas as pd
import requests
import json
import time
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f'Pandas: {pd.__version__}')
print(f'Requests: {requests.__version__}')

---
## 1. Основи HTTP та REST API

| Метод | Опис | DE-сценарій |
|-------|------|-------------|
| **GET** | Отримати дані | Завантажити список замовлень |
| **POST** | Створити ресурс | Відправити дані в API |
| **PUT** | Оновити ресурс | Оновити статус |
| **DELETE** | Видалити ресурс | Очистити тестові дані |

Більшість API повертають дані у форматі **JSON** — ідеально для Pandas.

In [ ]:
# Базовий GET-запит
url = 'https://jsonplaceholder.typicode.com/posts'
response = requests.get(url)

print(f'Status: {response.status_code}')
print(f'Content-Type: {response.headers["Content-Type"]}')
print(f'Response size: {len(response.content)} bytes')

# JSON -> Python list/dict
data = response.json()
print(f'Отримано записів: {len(data)}')
print(f'\nПерший запис:')
print(json.dumps(data[0], indent=2))

In [ ]:
# JSON прямо в DataFrame
df_posts = pd.DataFrame(data)
print('=== Posts DataFrame ===')
display(df_posts.head())
print(f'Shape: {df_posts.shape}')
print(f'Columns: {df_posts.columns.tolist()}')

---
## 2. Обробка вкладених JSON (Nested JSON)

API часто повертають вкладені об'єкти — `pd.json_normalize()` розгортає їх у flat таблицю.

In [ ]:
# API з вкладеним JSON
url = 'https://jsonplaceholder.typicode.com/users'
response = requests.get(url)
users = response.json()

# Подивимось структуру
print('Структура одного user:')
print(json.dumps(users[0], indent=2))

In [ ]:
# Проблема: pd.DataFrame() не розгортає вкладені об'єкти
df_users_bad = pd.DataFrame(users)
print('pd.DataFrame() — вкладені поля залишаються словниками:')
display(df_users_bad[['id', 'name', 'address', 'company']].head())

In [ ]:
# Рішення: pd.json_normalize()
df_users = pd.json_normalize(users)
print('pd.json_normalize() — всі поля розгорнуті:')
display(df_users.head())
print(f'\nКолонки: {df_users.columns.tolist()}')

In [ ]:
# Вибір потрібних колонок
users_flat = df_users[[
    'id', 'name', 'username', 'email',
    'address.city', 'address.street',
    'company.name'
]].rename(columns={
    'address.city': 'city',
    'address.street': 'street',
    'company.name': 'company'
})

print('Корисні дані про користувачів:')
display(users_flat)

In [ ]:
# json_normalize з розгортанням масивів (record_path)
# Приклад: API з масивом всередині
complex_data = [
    {
        'order_id': 1,
        'customer': {'name': 'Alice', 'email': 'alice@example.com'},
        'items': [
            {'product': 'Laptop', 'price': 1200, 'qty': 1},
            {'product': 'Mouse', 'price': 25, 'qty': 2}
        ]
    },
    {
        'order_id': 2,
        'customer': {'name': 'Bob', 'email': 'bob@example.com'},
        'items': [
            {'product': 'Monitor', 'price': 400, 'qty': 1}
        ]
    }
]

# Розгортаємо: record_path=масив, meta=поля на рівні вище
df_items = pd.json_normalize(
    complex_data,
    record_path='items',
    meta=[
        'order_id',
        ['customer', 'name'],
        ['customer', 'email']
    ]
)
print('Розгорнутий масив items:')
display(df_items)

---
## 3. API з авторизацією

Багато API вимагають токен або API key. Приклад з GitHub API.

In [ ]:
# GitHub API — публічний, без токена можна (але ліміт 60 req/h)
headers = {
    'Accept': 'application/vnd.github.v3+json',
    # 'Authorization': 'Bearer YOUR_GITHUB_TOKEN'  # для збільшення ліміту
}

# Отримаємо інформацію про репозиторій
url = 'https://api.github.com/repos/pandas-dev/pandas'
response = requests.get(url, headers=headers)

if response.status_code == 200:
    repo_data = response.json()
    repo_info = pd.json_normalize(repo_data)
    print('=== GitHub Repo Info ===')
    display(repo_info[[
        'name', 'full_name', 'description', 
        'stargazers_count', 'forks_count', 'open_issues_count',
        'language', 'created_at', 'updated_at'
    ]])
else:
    print(f'Помилка: {response.status_code}')

In [ ]:
# API з Bearer token (для прикладу — без реального токена)
TOKEN_EXAMPLE = """
headers = {
    'Authorization': 'Bearer YOUR_TOKEN',
    'Content-Type': 'application/json'
}
response = requests.get('https://api.example.com/data', headers=headers)
"""
print('Шаблон для API з Bearer токеном:')
print(TOKEN_EXAMPLE)

In [ ]:
# API Key у query параметрі
API_KEY_EXAMPLE = """
# Більшість публічних API використовують api_key як параметр
params = {
    'api_key': 'YOUR_API_KEY',
    'format': 'json'
}
response = requests.get('https://api.weather.gov/data', params=params)
"""
print('Шаблон для API Key:')
print(API_KEY_EXAMPLE)

---
## 4. Пагінація (Pagination)

Більшість API не повертають всі дані одразу — потрібно гортати сторінки.

In [ ]:
# Приклад пагінації: GitHub Issues API
# GitHub повертає по 30 items на сторінку, за замовчуванням

def fetch_all_issues(owner: str, repo: str, max_pages: int = 3) -> pd.DataFrame:
    """
    Збирає issues з GitHub API з пагінацією.
    """
    all_items = []
    page = 1

    while page <= max_pages:
        url = f'https://api.github.com/repos/{owner}/{repo}/issues'
        params = {'page': page, 'per_page': 30, 'state': 'all'}

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f'Помилка на сторінці {page}: {response.status_code}')
            break

        page_data = response.json()
        if not page_data:
            break  # більше даних немає

        all_items.extend(page_data)
        print(f'Сторінка {page}: отримано {len(page_data)} issues')

        # Link header для визначення наступної сторінки
        if 'next' not in response.links:
            break

        page += 1
        time.sleep(0.5)  # rate limiting

    if all_items:
        df = pd.json_normalize(all_items)
        return df
    return pd.DataFrame()

# Зберемо 2 сторінки issues з pandas (щоб не перевищити ліміт)
df_issues = fetch_all_issues('pandas-dev', 'pandas', max_pages=2)
print(f'\nВсього зібрано: {len(df_issues)} issues')
if len(df_issues) > 0:
    display(df_issues[['number', 'title', 'state', 'created_at', 'user.login']].head())

In [ ]:
# Універсальна функція пагінації для різних API
def paginate_api(base_url: str, params: dict = None, max_pages: int = 5,
                 page_param: str = 'page', per_page_param: str = 'per_page',
                 per_page: int = 100, headers: dict = None) -> pd.DataFrame:
    """
    Універсальна функція для збору даних з API з пагінацією.
    """
    all_data = []
    params = params or {}
    params[per_page_param] = per_page

    for page in range(1, max_pages + 1):
        params[page_param] = page

        try:
            response = requests.get(base_url, params=params, headers=headers, timeout=30)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f'Помилка на сторінці {page}: {e}')
            break

        if not data:
            break

        all_data.extend(data if isinstance(data, list) else [data])
        print(f'Сторінка {page}: {len(data) if isinstance(data, list) else 1} записів')
        time.sleep(0.3)

    if all_data:
        return pd.json_normalize(all_data)
    return pd.DataFrame()

print('Функція paginate_api() готова')

In [ ]:
# Приклад роботи з Nationalize API (без авторизації)
# Визначає національність за ім'ям
names = ['alice', 'bob', 'charlie', 'diana', 'eve', 'frank', 'grace', 'ivan', 'petro', 'olga']

results = []
for name in names:
    response = requests.get(f'https://api.nationalize.io?name={name}')
    if response.status_code == 200:
        data = response.json()
        results.append(data)
    time.sleep(0.2)

df_nationalize = pd.json_normalize(results)
print('=== Nationalize API ===')
display(df_nationalize[['name', 'country']].head())

# Розгорнемо country (масив)
df_countries = pd.json_normalize(
    results,
    record_path='country',
    meta=['name']
)
print('\n=== Розгорнуті країни ===')
display(df_countries.sort_values('probability', ascending=False).head(10))

---
## 5. POST-запити та відправка даних

In [ ]:
# POST — створення ресурсу
new_post = {
    'title': 'New DE Post',
    'body': 'Pandas + API is powerful',
    'userId': 1
}

response = requests.post(
    'https://jsonplaceholder.typicode.com/posts',
    json=new_post
)

if response.status_code == 201:
    created = response.json()
    print('POST успішний! Створено:')
    print(json.dumps(created, indent=2))
else:
    print(f'Помилка: {response.status_code}')

In [ ]:
# POST з DataFrame -> JSON
df_to_send = pd.DataFrame([
    {'title': 'Report 1', 'body': 'Q1 results', 'userId': 1},
    {'title': 'Report 2', 'body': 'Q2 results', 'userId': 2},
])

payload = df_to_send.to_dict(orient='records')
print('Payload для POST:')
print(json.dumps(payload, indent=2))

# Відправити всі записи
for record in payload:
    response = requests.post('https://jsonplaceholder.typicode.com/posts', json=record)
    print(f'Відправлено: {record["title"]} -> status {response.status_code}')

---
## 6. Збереження API-даних

Data Engineer має зберігати отримані дані в файли або БД.

In [ ]:
os.makedirs('demo_data', exist_ok=True)

# Отримаємо коментарі
response = requests.get('https://jsonplaceholder.typicode.com/comments')
df_comments = pd.DataFrame(response.json())
print(f'Отримано {len(df_comments)} коментарів')

# Збереження в різні формати
df_comments.to_csv('demo_data/comments.csv', index=False)
print('Збережено CSV: demo_data/comments.csv')

df_comments.to_parquet('demo_data/comments.parquet', index=False)
print('Збережено Parquet: demo_data/comments.parquet')

df_comments.to_json('demo_data/comments.json', orient='records', lines=True)
print('Збережено JSON: demo_data/comments.json')

In [ ]:
# Збереження в SQLite
from sqlalchemy import create_engine

engine = create_engine('sqlite:///demo_data/api_data.db')
df_comments.to_sql('comments', engine, if_exists='replace', index=False)
print('Збережено в SQLite: demo_data/api_data.db -> таблиця comments')

# Перевірка
check = pd.read_sql('SELECT COUNT(*) AS cnt FROM comments', engine)
print(f'Записів у SQLite: {check["cnt"].values[0]}')

---
## 7. Обробка помилок та Rate Limiting

В реальному DE API можуть падати або обмежувати швидкість запитів.

In [ ]:
# Патерн: retry з exponential backoff
import time

def fetch_with_retry(url: str, max_retries: int = 3) -> requests.Response:
    """
    GET-запит з повторними спробами при помилці.
    """
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 429:  # Too Many Requests
                wait = int(response.headers.get('Retry-After', 2 ** attempt))
                print(f'Rate limited. Чекаємо {wait}с...')
                time.sleep(wait)
                continue
            response.raise_for_status()
            return response
        except requests.exceptions.Timeout:
            print(f'Timeout (attempt {attempt}/{max_retries})')
        except requests.exceptions.RequestException as e:
            print(f'Помилка (attempt {attempt}/{max_retries}): {e}')

        if attempt < max_retries:
            time.sleep(2 ** attempt)  # exponential backoff: 2, 4, 8... секунд

    raise Exception(f'Не вдалося отримати {url} після {max_retries} спроб')

print('fetch_with_retry() готова до використання')

In [ ]:
# Тест retry
try:
    response = fetch_with_retry('https://jsonplaceholder.typicode.com/posts/1')
    print(f'Успіх! Отримано post #{response.json()["id"]}')
except Exception as e:
    print(f'Помилка: {e}')

In [ ]:
# Логування API-запитів (для моніторингу)
def log_api_call(url: str, status: int, duration: float, log_file: str = 'api_log.csv'):
    """
    ts, url, status, duration
    """
    log_entry = pd.DataFrame([{
        'timestamp': datetime.now().isoformat(),
        'url': url,
        'status': status,
        'duration_ms': round(duration * 1000, 2)
    }])

    if os.path.exists(log_file):
        log_entry.to_csv(log_file, mode='a', header=False, index=False)
    else:
        log_entry.to_csv(log_file, index=False)

# Приклад логування
start = time.time()
r = requests.get('https://jsonplaceholder.typicode.com/todos/1')
log_api_call(r.url, r.status_code, time.time() - start)

log_df = pd.read_csv('api_log.csv')
print('API Log:')
display(log_df)

---
## 8. ETL-пайплайн: API → Transform → DWH

Типовий DE-сценарій: збираємо дані з API, трансформуємо, зберігаємо.

In [ ]:
# Повний ETL пайплайн
def api_to_dwh_pipeline(api_url: str, db_path: str = 'demo_data/api_dwh.db'):
    """
    ETL: Extract з API -> Transform в Pandas -> Load в SQLite
    """
    print('=== ETL Pipeline Started ===')

    # --- Extract ---
    print(f'[EXTRACT] {api_url}')
    response = requests.get(api_url)
    response.raise_for_status()
    raw_data = response.json()
    print(f'[EXTRACT] Отримано {len(raw_data)} записів')

    # --- Transform ---
    print('[TRANSFORM] Початок трансформації')
    df = pd.DataFrame(raw_data)

    # datetime
    if 'created_at' in df.columns:
        df['created_at'] = pd.to_datetime(df['created_at'], format='mixed', errors='coerce')
    if 'updated_at' in df.columns:
        df['updated_at'] = pd.to_datetime(df['updated_at'], format='mixed', errors='coerce')

    # очищення
    df = df.dropna(how='all')

    # нові колонки
    if 'id' in df.columns:
        df['etl_loaded_at'] = datetime.now().isoformat()

    print(f'[TRANSFORM] Готово: {df.shape[0]} рядків, {df.shape[1]} колонок')

    # --- Load ---
    print(f'[LOAD] Запис в {db_path}')
    engine = create_engine(f'sqlite:///{db_path}')
    table_name = api_url.rstrip('/').split('/')[-1]
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f'[LOAD] Таблиця "{table_name}" створена')

    print('=== ETL Pipeline Completed ===')
    return df

# Запуск
df_etl_result = api_to_dwh_pipeline('https://jsonplaceholder.typicode.com/todos')
display(df_etl_result.head())

In [ ]:
# Перевірка, що дані в DWH
dwh_engine = create_engine('sqlite:///demo_data/api_dwh.db')
df_check = pd.read_sql('SELECT * FROM todos LIMIT 10', dwh_engine)
print('Дані в DWH:')
display(df_check)

---
## 9. Робота з реальними API (приклади)

API які можна використовувати без авторизації.

In [ ]:
# 1. Open Library API (книги)
response = requests.get('https://openlibrary.org/search.json?q=data+engineering&limit=5')
if response.status_code == 200:
    data = response.json()
    df_books = pd.json_normalize(data['docs'])
    print('=== Open Library: Data Engineering books ===')
    display(df_books[['title', 'author_name', 'first_publish_year', 'language']].head())

In [ ]:
# 2. CoinDesk Bitcoin Price Index
response = requests.get('https://api.coindesk.com/v1/bpi/currentprice.json')
if response.status_code == 200:
    data = response.json()
    bpi = pd.json_normalize(data['bpi']).T.reset_index()
    bpi.columns = ['currency', 'rate', 'description', 'rate_float']
    print('=== Bitcoin Price Index ===')
    display(bpi[['currency', 'rate_float']])

In [ ]:
# 3. NASA API (потрібен API key, але можна 'DEMO_KEY')
response = requests.get('https://api.nasa.gov/planetary/apod?api_key=DEMO_KEY&count=3')
if response.status_code == 200:
    data = response.json()
    df_nasa = pd.json_normalize(data)
    print('=== NASA APOD ===')
    display(df_nasa[['title', 'date', 'explanation']].head())

In [ ]:
# 4. Random User Generator
response = requests.get('https://randomuser.me/api/?results=5')
if response.status_code == 200:
    data = response.json()
    df_users = pd.json_normalize(data['results'])
    print('=== Random Users ===')
    display(df_users[[
        'name.first', 'name.last', 'email', 
        'location.city', 'location.country',
        'dob.age'
    ]].head())

---
## 10. Асинхронні запити (для продуктивності)

Коли потрібно зробити сотні запитів — `aiohttp` швидше за `requests`.

In [ ]:
# Синхронний підхід (base line)
import time

urls = [f'https://jsonplaceholder.typicode.com/posts/{i}' for i in range(1, 11)]

start = time.time()
sync_results = []
for url in urls:
    response = requests.get(url)
    sync_results.append(response.json())
sync_time = time.time() - start

print(f'Синхронно (10 запитів): {sync_time:.2f}с')

In [ ]:
# Асинхронний підхід з aiohttp
try:
    import aiohttp
    import asyncio

    async def fetch_async(session, url):
        async with session.get(url) as response:
            return await response.json()

    async def fetch_all_async(urls):
        async with aiohttp.ClientSession() as session:
            tasks = [fetch_async(session, url) for url in urls]
            return await asyncio.gather(*tasks)

    start = time.time()
    async_results = asyncio.run(fetch_all_async(urls))
    async_time = time.time() - start

    print(f'Асинхронно (10 запитів): {async_time:.2f}с')
    print(f'Прискорення: {sync_time / async_time:.1f}x')

    # Перетворюємо в DataFrame
    df_async = pd.DataFrame(async_results)
    print(f'\nAsync DataFrame: {df_async.shape}')
    display(df_async[['id', 'title']].head())

except ImportError:
    print('aiohttp не встановлено. Встановіть: pip install aiohttp')

---
## 11. Практичні завдання

1. **Отримайте** список користувачів з `https://jsonplaceholder.typicode.com/users`
2. **Розгорніть** вкладений JSON (address, company) через `json_normalize`
3. **Створіть** DataFrame з полями: id, name, email, city, company_name
4. **Збережіть** результат у CSV, JSON, Parquet
5. **Напишіть** функцію для збору всіх постів (100) з пагінацією
6. (Бонус) **Використайте** реальне API (Open Library, NASA, CoinDesk) і зробіть звіт

In [ ]:
# === Розв'язання ===

# 1-3. Отримуємо та обробляємо users
response = requests.get('https://jsonplaceholder.typicode.com/users')
users_raw = response.json()
df_users_full = pd.json_normalize(users_raw)

users_report = df_users_full[[
    'id', 'name', 'email',
    'address.city', 'company.name'
]].rename(columns={
    'address.city': 'city',
    'company.name': 'company'
})

print('=== Users Report ===')
display(users_report)

In [ ]:
# 4. Збереження
os.makedirs('demo_data', exist_ok=True)
users_report.to_csv('demo_data/users.csv', index=False)
users_report.to_json('demo_data/users.json', orient='records', indent=2)
users_report.to_parquet('demo_data/users.parquet', index=False)

print('Файли збережено:')
for f in ['demo_data/users.csv', 'demo_data/users.json', 'demo_data/users.parquet']:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f'  {f}: {size} bytes')

In [ ]:
# 5. Збір всіх постів через пагінацію
def fetch_all_posts():
    all_posts = []
    for page in range(1, 4):  # 100 posts / 50 per_page = 2 pages, +1 запас
        url = 'https://jsonplaceholder.typicode.com/posts'
        params = {'_page': page, '_limit': 50}
        response = requests.get(url, params=params)
        posts = response.json()
        if not posts:
            break
        all_posts.extend(posts)
        print(f'Page {page}: {len(posts)} posts')
    return pd.DataFrame(all_posts)

df_all_posts = fetch_all_posts()
print(f'\nВсього постів: {len(df_all_posts)}')
display(df_all_posts.head())

---
## Висновки

| Техніка | Інструмент | Коли використовувати |
|---------|-----------|---------------------|
| HTTP-запит | `requests.get()` | Основний спосіб отримання даних |
| JSON -> DataFrame | `pd.DataFrame(response.json())` | Простий JSON (flat) |
| Вкладений JSON | `pd.json_normalize()` | API з nested objects |
| Масиви в JSON | `json_normalize(record_path=..., meta=...)` | Orders з items |
| Пагінація | Цикл з `page` параметром | Багато сторінок даних |
| Retry | `time.sleep(2**attempt)` | Нестабільні API |
| Async | `aiohttp` + `asyncio` | 100+ швидких запитів |
| Логування | CSV з timestamp | Моніторинг ETL |

> **Порада DE**: завжди обробляйте помилки, додавайте retry-логіку і логування. API можуть бути нестабільними, і ваш пайплайн має це витримувати.